# Day 35：LlamaIndex 框架核心抽象与 LangChain 深度对比

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第三阶段 Day 35！今天我们将实战 LlamaIndex 的五大核心抽象：`SimpleDirectoryReader`、`Node`、`VectorStoreIndex`、`QueryEngine` 与 `ChatEngine`，并通过 `compare_frameworks()` 输出 LangChain 与 LlamaIndex 的选型与融合架构矩阵（`v0.3.0-day35`）。

## 1. 环境初始化与导入 `project_iterations/day_35` 模块
定位仓库根目录并将 `project_iterations/day_35` 加入 `sys.path`，导入 LlamaIndex 风格核心组件。

In [1]:
import sys
from pathlib import Path
import tempfile

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day35_dir = repo_root / "project_iterations" / "day_35"
if str(day35_dir) not in sys.path:
    sys.path.insert(0, str(day35_dir))

from zhishu_agent import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    compare_frameworks,
    format_framework_comparison,
)

print("已加载模块路径:", day35_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_35


## 2. `SimpleDirectoryReader` 与 `VectorStoreIndex.from_documents()`
使用临时目录模拟企业知识库，加载文档并构建带 `SOURCE / PREVIOUS / NEXT` 关系链的 `Node` 向量索引。

In [2]:
tmpdir = tempfile.TemporaryDirectory()
kb_dir = Path(tmpdir.name)
(kb_dir / "hr_leave.md").write_text(
    "星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天。"
    "申请年假需提前3个工作日在 OA 系统提交主管审批。",
    encoding="utf-8",
)
(kb_dir / "travel_policy.md").write_text(
    "差旅报销标准：一线城市酒店住宿报销上限每晚600元，二线城市每晚450元，需提供电子发票。",
    encoding="utf-8",
)
(kb_dir / "redis_ops.txt").write_text(
    "生产环境 Redis 集群严禁执行 KEYS * 命令，排查键空间请使用 SCAN 分批遍历。",
    encoding="utf-8",
)

docs = SimpleDirectoryReader(kb_dir).load_data()
print(f"加载文档数量: {len(docs)}")
for d in docs:
    print(f" - {d.doc_id} | {d.metadata['file_name']}")

index = VectorStoreIndex.from_documents(docs, chunk_size=45, chunk_overlap=10)
print(f"生成 Node 总数: {len(index.nodes)}")
print("首个 Node 关系链:", index.nodes[0].relationships)

加载文档数量: 3
 - doc-8eac0ffbff19 | hr_leave.md
 - doc-5d480568ee87 | redis_ops.txt
 - doc-449db404170a | travel_policy.md
生成 Node 总数: 5
首个 Node 关系链: {'SOURCE': 'doc-8eac0ffbff19', 'NEXT': 'doc-8eac0ffbff19-node-1'}


## 3. `QueryEngine`：无状态单轮精准问答与三种响应模式
对比 `compact`、`refine`、`tree_summarize` 三种响应合成策略，并查看检索溯源节点。

In [3]:
query = "生产环境 Redis 集群禁止执行什么命令？"
for mode in ("compact", "refine", "tree_summarize"):
    qe = index.as_query_engine(similarity_top_k=2, response_mode=mode)
    res = qe.query(query)
    print(f"[{mode}] {res.response}")

print("\n--- 溯源节点明细 ---")
print(index.as_query_engine(similarity_top_k=2).query(query).format_sources())

[compact] 根据知识库检索结果：生产环境 Redis 集群严禁执行 KEYS * 命令，排查键空间请使用 SCAN 分批遍。
[refine] [Refine 综合答案 | 主来源: redis_ops.txt] 生产环境 Redis 集群严禁执行 KEYS * 命令，排查键空间请使用 SCAN 分批遍。
[tree_summarize] [TreeSummarize 汇总 | 涉及文档: redis_ops.txt] 生产环境 Redis 集群严禁执行 KEYS * 命令，排查键空间请使用 SCAN 分批遍。

--- 溯源节点明细 ---
[1] (redis_ops.txt, score=0.5898) 生产环境 Redis 集群严禁执行 KEYS * 命令，排查键空间请使用 SCAN 分批遍...
[2] (hr_leave.md, score=0.0000) 星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天。...


## 4. `ChatEngine`：有状态多轮对话与指代追问改写
演示用户先问年假天数，再追问“申请它需要提前几天？”，验证 `ChatEngine` 的上下文改写与 `reset()` 重置能力。

In [4]:
chat_engine = index.as_chat_engine(similarity_top_k=2)
t1 = chat_engine.chat("星辰数科入职满1年员工有多少天年假？")
print("Turn 1:", t1.response)

t2 = chat_engine.chat("申请它需要提前几天？")
print("Turn 2 改写检索词:", t2.metadata["condensed_query"])
print("Turn 2:", t2.response)
print("当前历史轮数:", t2.metadata["history_turns"])

chat_engine.reset()
print("重置后历史记录数:", len(chat_engine.chat_history))
tmpdir.cleanup()

Turn 1: 根据知识库检索结果：星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天。
Turn 2 改写检索词: 星辰数科入职满1年员工有多少天年假？ 申请它需要提前几天？
Turn 2: 根据知识库检索结果：星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天。申请年假需提前3个工作日在 OA 系统提交主管审批。满10年享有10天。
当前历史轮数: 2
重置后历史记录数: 0


## 5. LangChain vs LlamaIndex 架构对比与场景选型
调用 `compare_frameworks()` 与 `format_framework_comparison()` 输出对比矩阵与混合架构推荐。

In [5]:
print(format_framework_comparison(use_case="hybrid"))

for case in ("enterprise_rag", "multi_agent_workflow", "hybrid"):
    rec = compare_frameworks(use_case=case)["recommendation"]
    print(f"\n[{case}] 推荐: {rec['recommended']}")

=== LangChain vs LlamaIndex 框架对比矩阵 ===
- [核心定位]
  * LangChain : 通用 LLM 应用与复杂 Agent 编排框架（Chains / LCEL / LangGraph）
  * LlamaIndex: 以数据为中心的 RAG 与知识索引框架（Data Framework for LLM）
- [核心抽象]
  * LangChain : Runnable、PromptTemplate、Chain、Tool、AgentExecutor、StateGraph
  * LlamaIndex: Document、Node、VectorStoreIndex、Retriever、QueryEngine、ChatEngine
- [数据接入与索引]
  * LangChain : DocumentLoader + TextSplitter + VectorStore，需手动串联切分与入库流水线
  * LlamaIndex: SimpleDirectoryReader + NodeParser + VectorStoreIndex.from_documents() 一键索引
- [检索与响应合成]
  * LangChain : Retriever 输出 Document 列表，需自行拼接 Prompt 与上下文压缩逻辑
  * LlamaIndex: 内置 QueryEngine（compact/refine/tree_summarize）与 ChatEngine 多轮改写
- [Agent 与复杂工作流]
  * LangChain : 生态最成熟，LangGraph 原生支持循环图、人工介入（HITL）与状态持久化
  * LlamaIndex: 支持 FunctionAgent 与 Workflows，但在复杂多智能体编排上不如 LangGraph 灵活
架构师总结: 二者并非非此即彼：知识检索底座优先 LlamaIndex，复杂多步 Agent 编排优先 LangChain/LangGraph。
场景推荐 (hybrid): LlamaIndex (检索索引层) + LangChain/LangGraph (Agent 编排层) — 生产级企业智能体最佳实践：用 LlamaIndex 管理高质量文档索引与 Qu